In [2]:
import random
import json
import math
import numpy as np

from typing import Dict, Any, Sequence, Optional 
from dataclasses import dataclass

from importnb import Notebook

with Notebook():
    # from LabTrajectory_RandomWalk import simulate_viewport_with_tiles
    from LabTrajectory_360Dataset import simulate_viewport_with_tiles, load_all_yaws_pitches
    from LabTrajectory_Pantelis import load_trajectories, simulate_viewport

import os
import sys

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

import Common.config as config
import Common.datatypes as datatypes
import Common.utils as utils

import importlib

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(utils)

# filename_path_ = '/home/eduardo/Workspace/CacheVideoPredict360/Dataset/Trajectories'
filename_path_ = r'c:\Users\es25591\Workspace\CacheVideoPredict360\Dataset\Trajectories'


In [3]:
# -------------------------------
# Weibull parameterization
# -------------------------------
@dataclass(frozen=True)
class WeibullParams:
    alpha: float  # shape (k)
    beta: float   # scale (lambda) > 0
    gamma: float  # shift/location (theta), can be negative/positive

# Table I coefficients (alpha, beta, gamma) by category
# Source: Proactive Video Chunks Caching and Processing... (WCNC'19), Table I.  (α, β, γ)  [P-square ≈ 0.999]
# Note: keep category keys consistent in your dataset/taxonomy.


CATEGORIES = [
    "Gaming",
    "Comedy",
    "Entertainment",
    "Education",
    "News",
    "Science",
    "Music",
    "Autos",
    "Sports",
    "Film",
]
CATEGORY_WEIBULL: Dict[str, WeibullParams] = {
    "Gaming":        WeibullParams(1.98, 0.45,  0.0146),
    "Comedy":        WeibullParams(2.89, 0.65, -0.0250),
    "Entertainment": WeibullParams(2.41, 0.56, -0.0064),
    "Education":     WeibullParams(2.40, 0.54, -0.0104),
    "News":          WeibullParams(4.70, 0.95, -0.2980),
    "Science":       WeibullParams(2.53, 0.53,  0.0130),
    "Music":         WeibullParams(2.45, 0.51,  0.0178),
    "Autos":         WeibullParams(2.68, 0.58,  0.0016),
    "Sports":        WeibullParams(4.34, 0.92, -0.2670),
    "Film":          WeibullParams(2.32, 0.62,  0.0205),
}

def _safe_pow(x: float, p: float) -> float:
    if x < 0:
        # guard tiny negatives caused by floating error for even/real powers
        x = 0.0
    return x ** p

def weibull_survival(c: float, params: WeibullParams) -> float:
    if c <= params.gamma:
        return 1.0
    z = (c - params.gamma) / params.beta
    return math.exp(-_safe_pow(z, params.alpha))

def prob_drop_between(c0: float, c1: float, params: WeibullParams) -> float:
    s0, s1 = weibull_survival(c0, params), weibull_survival(c1, params)
    p = max(0.0, s0 - s1)
    return min(p, 1.0)

def decide_continue_by_gop(current_gop: int,
                           total_gops: int,
                           category: str,
                           rng: Optional[random.Random] = None) -> bool:

    params = CATEGORY_WEIBULL[category]
    c_g = current_gop / total_gops
    c_next = (current_gop + 1) / total_gops
    p_drop = prob_drop_between(c_g, c_next, params)

    return random.random() >= p_drop if rng else random.random() >= p_drop

In [ ]:
class UserRequestEvents:
    def __init__(
        self,
        n_nodes: int = 1,
        n_users: int = 1,
        step_size: float = 5.0,
        zipf_alpha: float = 1.0,
        n_videos: int = 100,
        n_gops: int = 60,
        n_layers: int = 1,
        n_tiles: int = 4,
        n: int = 4,
        m: int = 3,
        arrival_rate: float = 10.0,
        users_viewport_tiles: Optional[Sequence[Any]] = [],
        requested_videos: Optional[Sequence[Any]] = [],
        users_arrivals: Optional[Sequence[Any]] = []
    ):
        self.user_gop_counter = [0 for _ in range(n_users)]
        self.user_visited = [set() for _ in range(n_users)]
        
        # self.user_node_map = [random.randrange(n_nodes) for _ in range(n_users)]
        self.user_node_map = [i % n_nodes for i in range(n_users)]

        self.n_nodes = n_nodes
        self.n_users = n_users

        self.step_size = step_size
        self.zipf_alpha = zipf_alpha
        self.n_videos = n_videos
        self.n_gops = n_gops
        self.n_layers = n_layers
        self.n_tiles = n_tiles
        self.n = n
        self.m = m
        self.users_viewport_tiles = users_viewport_tiles
        self.requested_videos = requested_videos
        self.users_arrivals = users_arrivals
        self.arrival_rate = arrival_rate

        self.users_categories = {}
        self.trajectories = []

    def gen_request_for_user(
        self, 
        u_idx: int, 
        du_bitmaps,
        mec_bitmap
    ) -> Dict[str, Any]:

        p_idx = self.user_node_map[u_idx]
        video = self.requested_videos[u_idx]
        gop = self.user_gop_counter[u_idx]

        tiles = []
        for y in range(self.m):
            for x in range(self.n):
                is_in_du = du_bitmaps[p_idx][video][0][y * self.n + x][gop] == 1 if du_bitmaps is not None else False
                is_in_mec = mec_bitmap[video][0][y * self.n + x][gop] == 1 if mec_bitmap is not None else False

                alpha_p_u = 0 # Served by DU (Edge)
                alpha_M_u = 0 # Served by MEC (Regional)
                alpha_C_u = 0 # Served by Cloud

                if is_in_du:
                    alpha_p_u = 1
                elif is_in_mec:
                    alpha_M_u = 1
                else:
                    alpha_C_u = 1

                tile = {
                    "tile": y * self.n + x,
                    "layer": 0,
                    "size": 2e6 / self.n_tiles,
                    "events": {
                        "alpha_p_u": alpha_p_u, 
                        "alpha_M_u": alpha_M_u, 
                        "alpha_C_u": alpha_C_u, 
                        "beta_p_u": 0, 
                        "beta_M_u": 0
                    }
                }
                tiles.append(tile)

        # 2. Request Enhancement Layer (Layer 1)
        cur_viewport = self.users_viewport_tiles[u_idx][gop]
        for x, y in cur_viewport:
            is_in_du = du_bitmaps[p_idx][video][1][y * self.n + x][gop] == 1 if du_bitmaps is not None else False
            is_in_mec = mec_bitmap[video][1][y * self.n + x][gop] == 1 if mec_bitmap is not None else False

            alpha_p_u = 0
            alpha_M_u = 0
            alpha_C_u = 0

            if is_in_du:
                alpha_p_u = 1
            elif is_in_mec:
                alpha_M_u = 1
            else:
                alpha_C_u = 1

            tile = {
                "tile": y * self.n + x,
                "layer": 1,
                "size": 15e6 / self.n_tiles,
                "events": {
                    "alpha_p_u": alpha_p_u, 
                    "alpha_M_u": alpha_M_u, 
                    "alpha_C_u": alpha_C_u, 
                    "beta_p_u": 0, 
                    "beta_M_u": 0
                }
            }
            tiles.append(tile)

        cur_viewport = np.array(
            [y * self.n + x for x, y in cur_viewport], dtype=int
        )

        return {
            "gop": gop,
            "u": u_idx,
            "p": p_idx, 
            "video": video,
            "viewport": cur_viewport,
            "tiles": tiles,
            "base_req_init": (gop == 0)
        }

    def step(self, du_bitmaps, mec_bitmap, user_visited):
        reqs = []
        for i in range(self.n_users):
            if i in user_visited:
                continue

            will_continue = decide_continue_by_gop(
                self.user_gop_counter[i], 
                self.n_gops, 
                self.users_categories[i]
            ) if self.user_gop_counter[i] >= 1 else True
            
            if not will_continue and not self.user_is_done(i):
                self.user_gop_counter[i] = 0
                self.requested_videos[i] = self.zipf_sampler() - 1
                self.users_viewport_tiles[i] = simulate_viewport(
                    self.requested_videos[i],
                    self.trajectories
                )
                self.users_categories[i] = CATEGORIES[self.requested_videos[i] % len(CATEGORIES)]

            elif (
                self.users_arrivals[i] <= self.step_count and 
                self.user_gop_counter[i] < self.n_gops
            ):
                req = self.gen_request_for_user(i, du_bitmaps, mec_bitmap)
                reqs.append(req)
                self.user_gop_counter[i] += 1
                
        self.step_count += 1
        return reqs

    def step_i(self, u_idx, du_bitmaps, mec_bitmap):
        req = None

        will_continue = decide_continue_by_gop(
            self.user_gop_counter[u_idx], 
            self.n_gops, 
            self.users_categories[u_idx]
        ) if self.user_gop_counter[u_idx] >= 1 else True
        
        if not will_continue and not self.user_is_done(u_idx):
            self.user_gop_counter[u_idx] = 0
            self.requested_videos[u_idx] = self.zipf_sampler() - 1
            self.users_viewport_tiles[u_idx] = simulate_viewport(
                self.requested_videos[u_idx],
                self.trajectories
            )
            self.users_categories[u_idx] = CATEGORIES[self.requested_videos[u_idx] % len(CATEGORIES)]

        elif (
            self.users_arrivals[u_idx] <= self.step_count and 
            self.user_gop_counter[u_idx] < self.n_gops
        ):
            req = self.gen_request_for_user(u_idx, du_bitmaps, mec_bitmap)
            self.user_gop_counter[u_idx] += 1

        return req
    
    def user_is_done(self, u_id: int) -> bool:
        return self.user_gop_counter[u_id] >= self.n_gops

    def all_users_done(self) -> bool:
        return all([
            self.user_is_done(u_id) for u_id in range(self.n_users)
        ])

    def get_user_gop(self, u_id: int) -> int:
        return self.user_gop_counter[u_id]

    def reset(self, **kwargs):
        self.step_count = 0
        self.user_gop_counter = [0 for _ in range(self.n_users)]
        self.user_visited = [False for _ in range(self.n_users)]

        self.users_arrivals = utils.poisson_per_users(
            total_users=self.n_users,
            rate_per_minute=self.arrival_rate
        )
        self.zipf_sampler = datatypes.ZipfSampler( 
            total_videos=self.n_videos, 
            alpha=self.zipf_alpha,
            seed=None
        )
        self.requested_videos = [
            self.zipf_sampler() - 1 for _ in range(self.n_users)
        ]

        self.users_categories = {
            u: CATEGORIES[v % len(CATEGORIES)] for u, v in enumerate(self.requested_videos)
        }

        ### Load dataset based on Pantelis' traces  ###
        self.trajectories = load_trajectories(
            filename_path_,
        )
        self.users_viewport_tiles = []
        for i in range(self.n_users):
            video = self.requested_videos[i]
            viewport_tiles = simulate_viewport(
                video, 
                self.trajectories
            )

            viewport_tiles = viewport_tiles  # Repeat to reach n_gops
            self.users_viewport_tiles.append(viewport_tiles)
        
        ### Load and shuffle viewport traces so users get randomized trajectories
        # yaws, pitches = load_all_yaws_pitches(
        #     filename_path_
        # )
        # viewport_data = list(zip(yaws, pitches))
        # random.shuffle(viewport_data)
        
        # self.users_viewport_tiles = []
        # for yaw, pitch in viewport_data:
        #     viewport_tiles, _ = simulate_viewport_with_tiles(
        #         num_steps=len(yaw),
        #         n=self.n,
        #         fov_yaw=100,
        #         fov_pitch=50,
        #         yaws=yaw,
        #         pitches=pitch,
        #     )
        #     self.users_viewport_tiles.append(viewport_tiles)
        
        ### Load dataset based on the LabTrajectory_RandomWalk ###
        # self.users_viewport_tiles = []
        # for _ in range(self.n_users):
        #     _, _, viewport_tiles, _ = simulate_viewport_with_tiles(
        #         num_steps=self.n_gops,
        #         n=self.n,
        #         fov_yaw=90,
        #         fov_pitch=50,
        #         damping=0.99,
        #         step_size=self.step_size,
        #         start_yaw=180,
        #         start_pitch=0
        #     )
        #     self.users_viewport_tiles.append(viewport_tiles)
        ### End of dataset loading ###
        
        info = {
            "viewport_tiles": self.users_viewport_tiles,
            "requested_videos": self.requested_videos,
            "users_arrivals": self.users_arrivals,
            "users_requests": [],
            "user_request": None
        }
        
        return None, info

In [5]:
# Validation helpers
def validate_request_struct(req, num_tiles: int) -> bool:
    print(req)
    assert set([
        'gop','u','p','video','tiles'
    ]).issubset(req.keys()), 'Missing top-level keys'
    
    tiles = req['tiles']
    
    # Allow empty tiles for initial step (gop == 0), otherwise expect full base layer
    if len(tiles) > 0:
        assert len([
            t for t in tiles if t['layer'] == 0
        ]) == num_tiles, 'Base layer tiles count mismatch'
    
    for t in tiles:
        for k in ['tile','layer','size','events']:
            assert k in t, f'Missing tile key {k}'
        
        ev = t['events']
        
        for ek in ['alpha_p_u','alpha_M_u','alpha_C_u','beta_p_u','beta_M_u']:
            assert ek in ev, f'Missing event key {ek}'
    
    return True

In [6]:
if __name__ == '__main__':
    steps_to_run = 3
    n_users = 10
    step_size = 5.0
    zipf_alpha = 1.0
    n_videos = 100
    n_gops = 60
    n_layers = 2    # base + enhancement
    n = 4           # grid dimension (n x n)
    m = 3
    n_tiles = n * m # total tiles
    
    print('='*5, 'UserTileRequestEvents Test Harness', '='*5)
    print(f'Grid: {n}x{m} -> {n_tiles} tiles | Users: {n_users} | Layers: {n_layers}')
    print('Running steps...')

    user_env = UserRequestEvents(
        n=n,
        n_gops=n_gops,
        n_nodes=5,
        n_users=n_users,
        step_size=step_size,
        zipf_alpha=zipf_alpha,
        n_videos=n_videos,
        n_layers=n_layers,
        n_tiles=n_tiles
    )

    user_env.reset()
    user_env.users_arrivals[:] = 0

    cache_bitmap = np.zeros((
        user_env.n_videos, 
        user_env.n_layers,
        user_env.n_tiles, 
        user_env.n_gops
    ), dtype=np.int8)
    cache_bitmap[:, 0, :, :] = 1  # base layer cached for all videos

    all_requests = []
    for gop in range(steps_to_run):
        enh_layer_v0 = np.zeros(n_tiles, dtype=int)
        enh_layer_v1 = np.zeros(n_tiles, dtype=int)

        random_tile_indices = random.sample(range(n_tiles), 4)
        for idx in random_tile_indices:
            enh_layer_v0[idx] = 1

        random_tile_indices = random.sample(range(n_tiles), 4)
        for idx in random_tile_indices:
            enh_layer_v1[idx] = 1

        cache_bitmap[0, 1, :, gop] = enh_layer_v0  # video 0 enh layer
        cache_bitmap[1, 1, :, gop] = enh_layer_v1  # video 1 enh layer

        print(cache_bitmap[0, 1, :, gop])
        print(cache_bitmap[1, 1, :, gop])

        reqs = user_env.step(
            du_bitmaps=None,
            mec_bitmap=cache_bitmap
        )

        for r in reqs:
            validate_request_struct(r, n_tiles)
            enh_tiles = [t for t in r['tiles'] if t['layer'] == 1]
            print(f"  User {r['u']} Video {r['video']} GOP {r['gop']} EnhTiles={len(enh_tiles)}")

        all_requests.extend(reqs)

        # reqs = user_env.step(
        #     du_bitmaps=None,
        #     mec_bitmap=cache_bitmap
        # )

    print('\nSummary:')
    print(f'Total requests collected: {len(all_requests)}')
    base_sizes = set(t['size'] for r in all_requests for t in r['tiles'] if t['layer']==0)
    enh_sizes = set(t['size'] for r in all_requests for t in r['tiles'] if t['layer']==1)
    print(f'Base layer size values: {base_sizes}')
    print(f'Enh layer size values: {enh_sizes}')

    print('\nSample request (first user, first step):')
    # Print all requests in a readable format
    print(json.dumps(all_requests, indent=4))

===== UserTileRequestEvents Test Harness =====
Grid: 4x3 -> 12 tiles | Users: 10 | Layers: 2
Running steps...
[0 0 0 1 0 1 1 0 0 0 0 1]
[0 0 0 1 0 1 0 0 0 0 1 1]
{'gop': 0, 'u': 0, 'p': 0, 'video': 10, 'viewport': array([ 6,  7, 10, 11]), 'tiles': [{'tile': 0, 'layer': 0, 'size': 166666.66666666666, 'events': {'alpha_p_u': 0, 'alpha_M_u': 1, 'alpha_C_u': 0, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile': 1, 'layer': 0, 'size': 166666.66666666666, 'events': {'alpha_p_u': 0, 'alpha_M_u': 1, 'alpha_C_u': 0, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile': 2, 'layer': 0, 'size': 166666.66666666666, 'events': {'alpha_p_u': 0, 'alpha_M_u': 1, 'alpha_C_u': 0, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile': 3, 'layer': 0, 'size': 166666.66666666666, 'events': {'alpha_p_u': 0, 'alpha_M_u': 1, 'alpha_C_u': 0, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile': 4, 'layer': 0, 'size': 166666.66666666666, 'events': {'alpha_p_u': 0, 'alpha_M_u': 1, 'alpha_C_u': 0, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile': 5, 'layer': 0, 'size': 1666

TypeError: Object of type ndarray is not JSON serializable